# Fashion MNIST - Regularisering i CNN

Namn: Soroush Gholamreza

Fokusområde: Regularisering (Dropout och Early Stopping)

Syfte:
Att undersöka hur regularisering påverkar prestandan hos en CNN-modell för klassificering av klädesplagg i Fashion MNIST.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [4]:
import os
import gzip
import urllib.request
import numpy as np

DATA_DIR = "../data"

urls = {
    "train_images": "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/train-images-idx3-ubyte.gz",
    "train_labels": "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/train-labels-idx1-ubyte.gz",
    "test_images": "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/t10k-images-idx3-ubyte.gz",
    "test_labels": "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/t10k-labels-idx1-ubyte.gz"
}

os.makedirs(DATA_DIR, exist_ok=True)

for filename, url in urls.items():
    file_path = os.path.join(DATA_DIR, filename + ".gz")
    if not os.path.exists(file_path):
        print(f"Laddar ner {filename}...")
        urllib.request.urlretrieve(url, file_path)
    else:
        print(f"{filename} finns redan.")

def load_images(file_path):
    with gzip.open(file_path, "rb") as f:
        data = np.frombuffer(f.read(), np.uint8, offset=16)
    return data.reshape(-1, 28, 28)

def load_labels(file_path):
    with gzip.open(file_path, "rb") as f:
        data = np.frombuffer(f.read(), np.uint8, offset=8)
    return data

x_train = load_images(os.path.join(DATA_DIR, "train_images.gz"))
y_train = load_labels(os.path.join(DATA_DIR, "train_labels.gz"))
x_test = load_images(os.path.join(DATA_DIR, "test_images.gz"))
y_test = load_labels(os.path.join(DATA_DIR, "test_labels.gz"))

print("Träningsbilder:", x_train.shape)
print("Testbilder:", x_test.shape)
print("Träningsetiketter:", y_train.shape)
print("Testetiketter:", y_test.shape)

Laddar ner train_images...
Laddar ner train_labels...
Laddar ner test_images...
Laddar ner test_labels...
Träningsbilder: (60000, 28, 28)
Testbilder: (10000, 28, 28)
Träningsetiketter: (60000,)
Testetiketter: (10000,)


## Notering om datasetkälla

Enligt uppgiftsbeskrivningen skulle Fashion MNIST laddas via:

```python
keras.datasets.fashion_mnist
```

Vid försök att använda denna metod uppstod följande fel:

```text
403 Forbidden

Access denied.

We're sorry, but this service is not available in your location.
```

Fashion MNIST som tillhannahålls via Keras/TensorFlow hämtas från Google lagringsserver (`storage.googleapis.com`). Eftersom jag genomför arbetet från Iran var denna tjänst inte tillgänglig från min nuvarande plats.

Jag testade även att hämta Fashion MNIST via OpenML, men nedladdningen misslyckades på grund av ett tekniskt fel vid hämtning av datasetets metadata.

För att kunna genomföra uppgiften användes därför Fashion MNIST från det officiella GitHub-repot för Fashion MNIST, publicerat av Zalando Research som skapade datasetet.

Det dataset som används i projektet är fortfarande Fashion MNIST med:

- 70 000 bilder totalt
- 60 000 träningsbilder
- 10 000 testbilder
- 10 klasser av klädesplagg
- bildstorlek 28x28 pixlar i gråskala

Endast nedladdningskällan skiljer sig åt. Datasetets innehåll, etikettter och användningsområde är identiska med den som laddas via Keras.